In [ ]:
# INSTALAR LIBRERIAS
#!pip install -U langchain langchain-text-splitters langchain-chroma langchain-huggingface chromadb docx2txt sentence-transformers

In [1]:
import os

from docx import Document as WordDocument

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
RUTA_DOCUMENTOS = r"./documentos"

In [3]:
### FUNCION LEER WORD ###

def cargar_docx(ruta):

    docx = WordDocument(ruta)

    textos = []

    for parrafo in docx.paragraphs:

        texto = parrafo.text.strip()

        if texto:
            textos.append(texto)

    contenido = "\n".join(textos)

    return Document(
        page_content=contenido,
        metadata={
            "source": ruta,
            "nombre": os.path.basename(ruta),
            "tipo": "docx"
        }
    )

In [4]:
### CARGAR ARCHIVOS ###
documentos = []

for archivo in os.listdir(RUTA_DOCUMENTOS):

    if archivo.lower().endswith(".docx"):

        ruta = os.path.join(
            RUTA_DOCUMENTOS,
            archivo
        )

        documento = cargar_docx(ruta)

        documentos.append(documento)

print(
    f"Documentos cargados: {len(documentos)}"
)

Documentos cargados: 3


In [5]:
## REVISAR LOS .DOCX QUE ENCONTRÓ ##
for documento in documentos:

    print(
        documento.metadata["nombre"]
    )

Cultura_Chimu.docx
Cultura_Inca.docx
Historia_del_Peru.docx


In [6]:
### CREAR CHUNKING (PARTIR EL DOCUMENTO, 1000 Y 200 es lo más utilizado, quiere decir que lo va a partir cada 1000 caracteres tomando 200 del primer trozo y 200 del último)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(
    documentos
)

In [7]:
## COMPROBAR CANTIDAD DE CHUNKS ##
print(
    f"Total de chunks: {len(chunks)}"
)

Total de chunks: 21


In [8]:
## REVISAR LOS CHUNKS ##
for i, chunk in enumerate(chunks[:10]):

    print("=" * 100)

    print("CHUNK:", i)

    print(
        "DOCUMENTO:",
        chunk.metadata["nombre"]
    )

    print(
        "CARACTERES:",
        len(chunk.page_content)
    )

    print()

    print(chunk.page_content)

CHUNK: 0
DOCUMENTO: Cultura_Chimu.docx
CARACTERES: 841

La Cultura Chimú
El reino del gran Chimor en la costa norte del Perú
1. Origen y contexto histórico
La cultura Chimú se desarrolló en la costa norte del actual Perú entre aproximadamente los años 900 y 1470 d.C., sucediendo a la cultura Moche en la misma región. Según la tradición oral recogida por los cronistas españoles, el reino chimú —conocido como el Gran Chimor— fue fundado por un personaje mítico llamado Tacaynamo, quien habría llegado por mar y establecido su dominio en el valle de Moche.
Con el tiempo, el reino se expandió considerablemente bajo gobernantes posteriores hasta convertirse en el Estado más extenso y poderoso de la costa peruana antes de la llegada de los incas, llegando a controlar cerca de mil kilómetros de litoral, desde Tumbes por el norte hasta cerca de Lima por el sur.
2. Chan Chan: la capital de barro
CHUNK: 1
DOCUMENTO: Cultura_Chimu.docx
CARACTERES: 841

2. Chan Chan: la capital de barro
La capital d

In [9]:
for i, chunk in enumerate(chunks):

    chunk.metadata["chunk_id"] = i

In [10]:
# CADA CHUNK TIENE UN ID ##
print(chunks[0].metadata)

{'source': 'C:\\Users\\gehf2\\OneDrive\\Documentos\\AMORCITO\\Proyecto_Chat\\Documentos\\Cultura_Chimu.docx', 'nombre': 'Cultura_Chimu.docx', 'tipo': 'docx', 'chunk_id': 0}


In [11]:
## EMBEDDING LOCAL ##
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="documentos_word"
)

In [16]:
## BUSQUEDA SEMANTICA SOBRE DOCUMENTOS ##
pregunta = "que es chan chan"
resultados = vectorstore.similarity_search(
    pregunta,
    k=5
)

In [17]:
## 
for i, resultado in enumerate(resultados):

    print("=" * 100)

    print(
        f"RESULTADO {i + 1}"
    )

    print(
        "DOCUMENTO:",
        resultado.metadata["nombre"]
    )

    print(
        "CHUNK:",
        resultado.metadata["chunk_id"]
    )

    print()

    print(
        resultado.page_content
    )

RESULTADO 1
DOCUMENTO: Cultura_Chimu.docx
CHUNK: 5

A pesar de su incorporación al Tahuantinsuyo, la cultura chimú dejó una huella profunda en la costa norte peruana. Hoy, las ruinas de Chan Chan, declaradas Patrimonio Cultural de la Humanidad por la UNESCO en 1986, constituyen un testimonio monumental del ingenio urbanístico y artístico de este importante reino prehispánico.
RESULTADO 2
DOCUMENTO: Cultura_Chimu.docx
CHUNK: 1

2. Chan Chan: la capital de barro
La capital del reino chimú fue Chan Chan, ubicada en el valle de Moche, cerca de la actual ciudad de Trujillo. Se trata de la ciudad de adobe más grande de la América prehispánica y una de las más extensas del mundo antiguo construida con este material, con una superficie que superaba las veinte kilómetros cuadrados en su momento de mayor esplendor.
Chan Chan estaba organizada en nueve grandes recintos amurallados, conocidos como ciudadelas, cada uno asociado probablemente a un gobernante y funcionando como centro administrativo,

In [18]:
resultados = vectorstore.similarity_search_with_score(
    pregunta,
    k=5
)

In [19]:
for i, (resultado, score) in enumerate(resultados):

    print("=" * 100)

    print(
        f"RESULTADO {i + 1}"
    )

    print(
        "SCORE:",
        score
    )

    print(
        "DOCUMENTO:",
        resultado.metadata["nombre"]
    )

    print(
        "CHUNK:",
        resultado.metadata["chunk_id"]
    )

    print()

    print(
        resultado.page_content
    )

RESULTADO 1
SCORE: 0.9219830632209778
DOCUMENTO: Cultura_Chimu.docx
CHUNK: 5

A pesar de su incorporación al Tahuantinsuyo, la cultura chimú dejó una huella profunda en la costa norte peruana. Hoy, las ruinas de Chan Chan, declaradas Patrimonio Cultural de la Humanidad por la UNESCO en 1986, constituyen un testimonio monumental del ingenio urbanístico y artístico de este importante reino prehispánico.
RESULTADO 2
SCORE: 0.9752327799797058
DOCUMENTO: Cultura_Chimu.docx
CHUNK: 1

2. Chan Chan: la capital de barro
La capital del reino chimú fue Chan Chan, ubicada en el valle de Moche, cerca de la actual ciudad de Trujillo. Se trata de la ciudad de adobe más grande de la América prehispánica y una de las más extensas del mundo antiguo construida con este material, con una superficie que superaba las veinte kilómetros cuadrados en su momento de mayor esplendor.
Chan Chan estaba organizada en nueve grandes recintos amurallados, conocidos como ciudadelas, cada uno asociado probablemente a un 

In [22]:
### DEVUELVE LOS 2 RESULTADOS CON MEJOR SCORE ##

pregunta = "imperio inca"

resultados = vectorstore.similarity_search_with_score(
    pregunta,
    k=2
)

for i, (resultado, score) in enumerate(resultados, start=1):
    print("=" * 100)
    print(f"RESULTADO {i}")
    print(f"Distancia: {score:.4f}")
    print(f"Documento: {resultado.metadata['nombre']}")
    print(f"Chunk: {resultado.metadata['chunk_id']}")
    print(f"Contenido:\n{resultado.page_content}")

RESULTADO 1
Distancia: 0.7776
Documento: Historia_del_Peru.docx
Chunk: 15
Contenido:
2. El Tahuantinsuyo: el imperio incaico
Hacia el siglo XIII surgió, en el valle del Cusco, el pequeño reino inca que con el paso de los siglos se transformaría en el imperio más extenso de la América prehispánica: el Tahuantinsuyo. Bajo gobernantes como Pachacútec, Túpac Yupanqui y Huayna Cápac, el Estado inca se expandió mediante conquista militar y alianzas, llegando a abarcar territorios que hoy corresponden a Perú, Ecuador, Bolivia, el norte de Chile, el noroeste de Argentina y el sur de Colombia.
El imperio se organizó en cuatro suyos o regiones que convergían en el Cusco, su capital. Contaba con un eficiente sistema vial —el Qhapaq Ñan—, una administración basada en el quipu para el registro numérico, y una economía sustentada en el trabajo colectivo conocido como mita y en la reciprocidad andina. La llegada de los conquistadores españoles y la guerra civil entre Huáscar y Atahualpa debilitaron a